# Predictive AI Evaluation Challenge — Metadata-only Latent-Factor Model (Colab)

This notebook clones the competition repo, downloads the public HuggingFace response parquets, trains a metadata-only PyTorch latent-factor model, and runs the official-like validation harness across 3 seeds.

**Statistical form**
$$\eta_{m,b,c} = \mu + a_m + b_{b,c} + \frac{u_m \cdot v_{b,c}}{\sqrt{k}}$$
$$p_{m,b,c} = \sigma(\eta_{m,b,c})$$

The model intentionally **does not see `item_content`**. Headline baseline to beat: a no-cross-term logistic regression with mean log-likelihood $\approx -0.5224$.

**Recommended runtime**: A100 (Runtime → Change runtime type → GPU → A100). L4/T4 also work — the model is small and the bottleneck is data movement.

## 1. Environment + GPU info

In [ ]:
import os, sys, subprocess, json, time, shutil
from pathlib import Path

print('Python:', sys.version.split()[0])
try:
    import torch
    print('torch:', torch.__version__, 'cuda:', torch.version.cuda, 'cuda_available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        free, total = torch.cuda.mem_get_info(0)
        print(f'GPU memory: free={free/1e9:.2f}GB total={total/1e9:.2f}GB')
except ImportError:
    print('torch is not installed yet — will install in next cell.')
subprocess.run(['nvidia-smi'], check=False)

## 2. Install dependencies

Colab already ships `torch`, `pandas`, `numpy`, `scikit-learn`, `pyarrow`. We only need to ensure `huggingface_hub` and `datasets` are present for the data download. We also install `tqdm` (already there but pinned for safety).

In [ ]:
%pip install -q --upgrade huggingface_hub datasets pyarrow pandas numpy scikit-learn tqdm

## 3. Clone the competition repo

The repo bundles `validation_harness/`, `starting_kit/Model_Info/model_info.csv`, `starting_kit/benchmark_info/benchmark_info.csv`, and `Google_Collab_harness/` (this folder). It does **not** include the response parquets — those come from HuggingFace in the next step.

In [ ]:
REPO_URL = 'https://github.com/bwathomas/Prediction-Competition-321M.git'
REPO_DIR = Path('/content/Prediction-Competition-321M').resolve()
if REPO_DIR.exists():
    print(f'{REPO_DIR} already exists, pulling latest …')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

for sub in ['validation_harness', 'starting_kit/Model_Info', 'starting_kit/benchmark_info', 'Google_Collab_harness']:
    p = REPO_DIR / sub
    print(f'  {sub:40s} {"OK" if p.exists() else "MISSING"}')

GCH = REPO_DIR / 'Google_Collab_harness'
if str(GCH) not in sys.path:
    sys.path.insert(0, str(GCH))
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

## 4. Download the response parquets from HuggingFace

We download all `*.parquet` files from `aims-foundations/measurement-db` except the `*_traces.parquet` ones (different schema, not used). Total ≈ 1.5 GB.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

REPO_ID = 'aims-foundations/measurement-db'
DATA_DIR = REPO_DIR / 'starting_kit' / 'Data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

api = HfApi()
files = api.list_repo_files(repo_id=REPO_ID, repo_type='dataset')
wanted = [
    f for f in files
    if f.endswith('.parquet') and not f.endswith('_traces.parquet')
]
print(f'{len(wanted)} parquets to download')

for f in wanted:
    out = DATA_DIR / Path(f).name
    if out.exists() and out.stat().st_size > 0:
        continue
    print(f'  downloading {f} …', flush=True)
    p = hf_hub_download(repo_id=REPO_ID, filename=f, repo_type='dataset', local_dir=str(DATA_DIR), local_dir_use_symlinks=False)
    if Path(p).resolve() != out.resolve():
        try:
            shutil.copy2(p, out)
        except shutil.SameFileError:
            pass
print('done. files:')
subprocess.run(['ls', '-lh', str(DATA_DIR)], check=False)

## 5. Build the official item-cold-start split

Reuses `validation_harness/scripts/prepare_split.py` so the validation we report is exactly the official-like protocol.

In [ ]:
HARNESS_DIR = REPO_DIR / 'validation_harness'
SPLITS_DIR = HARNESS_DIR / 'splits' / 'v1'
if not (SPLITS_DIR / 'train.parquet').exists():
    subprocess.check_call([
        sys.executable,
        str(HARNESS_DIR / 'scripts' / 'prepare_split.py'),
        '--data-dir', str(DATA_DIR),
        '--out-dir',  str(SPLITS_DIR),
        '--val-fraction', '0.10',
        '--seed', '0',
    ])
else:
    print(f'Reusing existing split at {SPLITS_DIR}')
subprocess.run(['ls', '-lh', str(SPLITS_DIR)], check=False)

## 6. Train the latent-factor model + run official-like validation

This wraps everything: fit preprocessor on TRAIN ONLY, aggregate by (model, benchmark, condition) cells, train with AdamW + AMP + early stopping, package a submission folder, and run `validation_harness/scripts/run_validation.py` across seeds 0/1/2.

Defaults: `latent_dim=16`, `hidden_dim=256`, `num_layers=2`, `dropout=0.1`, `batch_size=65536`, `epochs=30`, `patience=5`.

In [ ]:
OUTPUT_DIR = REPO_DIR / 'outputs' / 'latent_factor'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, str(GCH / 'run_latent_factor_colab.py'),
    '--data-dir',                str(DATA_DIR),
    '--splits-dir',              str(SPLITS_DIR),
    '--model-info-csv',          str(REPO_DIR / 'starting_kit' / 'Model_Info' / 'model_info.csv'),
    '--benchmark-info-csv',      str(REPO_DIR / 'starting_kit' / 'benchmark_info' / 'benchmark_info.csv'),
    '--validation-harness-dir',  str(HARNESS_DIR),
    '--output-dir',              str(OUTPUT_DIR),
    '--latent-dim', '16',
    '--hidden-dim', '256',
    '--num-layers', '2',
    '--dropout',    '0.1',
    '--weight-decay','1e-4',
    '--batch-size', '65536',
    '--epochs',     '30',
    '--patience',   '5',
    '--official-seeds', '0', '1', '2',
    '--official-n', '5000',
    '--official-k', '5',
    '--logistic-baseline-ll', '-0.5224',
]
print('Running:', ' '.join(cmd))
import subprocess
subprocess.run(cmd, check=True)

## 7. (Optional) hyperparameter sweep

Runs the same script with `--sweep`, which iterates over `latent_dim ∈ {4, 8, 16, 32}`, `weight_decay ∈ {1e-4, 1e-3}`, `dropout ∈ {0.05, 0.1, 0.2}`. Each run trains from scratch; the best by full-val log-likelihood is saved as the canonical checkpoint and used for the official-like validation.

In [ ]:
# subprocess.run(cmd + ['--sweep'], check=True)

## 8. Read the metrics + interpret results

In [ ]:
import pandas as pd, json
summary = json.loads((OUTPUT_DIR / 'metrics.json').read_text())
print(json.dumps(summary, indent=2))

print('\nbaseline_comparison.csv:')
print(pd.read_csv(OUTPUT_DIR / 'baseline_comparison.csv').to_string(index=False))

print('\nofficial_seeds.csv:')
print(pd.read_csv(OUTPUT_DIR / 'official_seeds.csv').to_string(index=False))

if (OUTPUT_DIR / 'runs.csv').exists():
    print('\nruns.csv (top 10 by final_val_log_likelihood):')
    runs = pd.read_csv(OUTPUT_DIR / 'runs.csv')
    cols = ['run_idx', 'latent_dim', 'weight_decay', 'dropout', 'best_epoch',
            'final_train_log_likelihood', 'final_val_log_likelihood', 'wall_seconds']
    cols = [c for c in cols if c in runs.columns]
    print(runs[cols].sort_values('final_val_log_likelihood', ascending=False).head(10).to_string(index=False))

off_mean = summary.get('official_mean_log_likelihood')
imp = summary.get('improvement_vs_logistic_baseline')
if off_mean is not None and imp is not None:
    verdict = 'BEATS' if (imp or 0) > 0 else 'TRAILS'
    print(f'\nOfficial-like mean LL = {off_mean:+.4f}    {verdict} the logistic baseline by {imp:+.4f}')